In [ ]:
# MGMT Promoter Methylation Prediction using Peritumoral Radiomics
**Author:** Rafael Boava Souza, MD (Radiology Resident - UNIFESP)

This notebook contains the complete pipeline for predicting MGMT methylation status using the BraTS 2021 dataset, focusing on peritumoral edema (habitat radiomics).

!pip install pyradiomics SimpleITK xgboost
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Path to your master CSV in Google Drive
path_master = '/content/drive/MyDrive/Projeto_BraTS_Radiomica/base_treinamento_final.csv'
df = pd.read_csv(path_master)

# Features (X) and Target (y)
X = df.drop(columns=['ID', 'MGMT_value'])
y = df['MGMT_value']

# Train/Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Feature Scaling - Essential for SVM
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

# Models to evaluate
models = {
    "Random Forest": RandomForestClassifier(n_estimators=500, min_samples_leaf=5, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42),
    "SVM (Optimal)": SVC(probability=True, kernel='rbf', random_state=42)
}

print("Running benchmarking...")
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    probs = model.predict_proba(X_test_scaled)[:, 1]
    print(f"🔹 {name} AUC: {roc_auc_score(y_test, probs):.3f}")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report

# 1. DATA PREPARATION
# Ensuring the same split used in your analysis
path_master = '/content/drive/MyDrive/Projeto_BraTS_Radiomica/base_treinamento_final.csv'
df = pd.read_csv(path_master)

X = df.drop(columns=['ID', 'MGMT_value'])
y = df['MGMT_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scaling features (Mandatory for SVM performance)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- PLOT 1: MGMT STATUS DISTRIBUTION ---
plt.figure(figsize=(8, 6))
sns.countplot(x=y, palette='viridis')
plt.title('Distribution of MGMT Methylation Status', fontsize=14, fontweight='bold')
plt.xlabel('MGMT Status (0: Unmethylated, 1: Methylated)', fontsize=12)
plt.ylabel('Number of Patients', fontsize=12)
plt.xticks([0, 1], ['Unmethylated (0)', 'Methylated (1)'])
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.savefig('mgmt_distribution.png', dpi=300, bbox_inches='tight')
plt.show()
files.download('mgmt_distribution.png')

# --- PLOT 2: MODEL BENCHMARKING (AUC COMPARISON) ---
models = {
    "Random Forest": RandomForestClassifier(n_estimators=500, min_samples_leaf=5, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42),
    "SVM (Champion)": SVC(probability=True, kernel='rbf', random_state=42)
}

auc_results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    probs = model.predict_proba(X_test_scaled)[:, 1]
    auc_results[name] = roc_auc_score(y_test, probs)

plt.figure(figsize=(10, 6))
colors = ['#bdc3c7', '#bdc3c7', '#27ae60'] # Green highlight for the best model
plt.bar(auc_results.keys(), auc_results.values(), color=colors)
plt.axhline(y=0.5, color='#e74c3c', linestyle='--', label='Random Chance (0.5)')
plt.ylim(0.4, 0.7)
plt.title('Machine Learning Benchmarking (AUC Score)', fontsize=14, fontweight='bold')
plt.ylabel('AUC Score', fontsize=12)
plt.legend(loc='upper left')
plt.savefig('model_benchmarking.png', dpi=300, bbox_inches='tight')
plt.show()
files.download('model_benchmarking.png')

# --- PLOT 3: RADIOMIC FEATURE IMPORTANCE ---
# Using Random Forest to explain feature relevance
rf_explainer = models["Random Forest"]
importances = rf_explainer.feature_importances_
indices = np.argsort(importances)[-15:]

plt.figure(figsize=(10, 8))
plt.title('Top 15 Most Predictive Radiomic Features', fontsize=14, fontweight='bold')
plt.barh(range(len(indices)), importances[indices], color='#3498db', align='center')
plt.yticks(range(len(indices)), [X.columns[i] for i in indices])
plt.xlabel('Relative Feature Importance', fontsize=12)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()
files.download('feature_importance.png')

# --- PLOT 4: FINAL ROC CURVE (SVM) ---
svm_model = models["SVM (Champion)"]
svm_probs = svm_model.predict_proba(X_test_scaled)[:, 1]
fpr, tpr, _ = roc_curve(y_test, svm_probs)
final_auc = roc_auc_score(y_test, svm_probs)

plt.figure(figsize=(7, 7))
plt.plot(fpr, tpr, color='#f39c12', lw=3, label=f'SVM Classifier (AUC = {final_auc:.2f})')
plt.plot([0, 1], [0, 1], color='#2c3e50', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
plt.ylabel('True Positive Rate (Sensitivity)', fontsize=12)
plt.title('Receiver Operating Characteristic (ROC) Curve', fontsize=14, fontweight='bold')
plt.legend(loc="lower right")
plt.grid(alpha=0.2)
plt.savefig('roc_curve_final.png', dpi=300, bbox_inches='tight')
plt.show()
files.download('roc_curve_final.png')

# --- PLOT 5: CONFUSION MATRIX (CLINICAL UTILITY) ---
svm_preds = svm_model.predict(X_test_scaled)
cm = confusion_matrix(y_test, svm_preds)

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlGnBu',
            xticklabels=['Predicted: Unmethylated', 'Predicted: Methylated'],
            yticklabels=['Actual: Unmethylated', 'Actual: Methylated'])
plt.title('Confusion Matrix: SVM Clinical Performance', fontsize=14, fontweight='bold')
plt.ylabel('Actual Label (Ground Truth)', fontsize=12)
plt.xlabel('AI Model Prediction', fontsize=12)
plt.savefig('confusion_matrix_final.png', dpi=300, bbox_inches='tight')
plt.show()
files.download('confusion_matrix_final.png')

print("\n🚀 All plots have been generated and downloaded. Your GitHub 'results' folder is ready!")